<a href="https://colab.research.google.com/github/cojocarucosmin/AICourseDev/blob/main/Metadata_Document_Extractor_using_OCR_an_ChatGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Parsing and extracting inforrmation from documents using OCR an ChatGPT**

### **Install required libraries**

In [12]:
# 🚫 DO NOT EDIT – install required libraries
!pip install -q pymupdf pymupdf4llm pytesseract pillow openai gradio tiktoken
!apt-get install -y -qq tesseract-ocr
!apt-get install -y -qq libtesseract-dev

Selecting previously unselected package libarchive-dev:amd64.
(Reading database ... 126315 files and directories currently installed.)
Preparing to unpack .../libarchive-dev_3.6.0-1ubuntu1.3_amd64.deb ...
Unpacking libarchive-dev:amd64 (3.6.0-1ubuntu1.3) ...
Selecting previously unselected package libleptonica-dev.
Preparing to unpack .../libleptonica-dev_1.82.0-3build1_amd64.deb ...
Unpacking libleptonica-dev (1.82.0-3build1) ...
Selecting previously unselected package libtesseract-dev:amd64.
Preparing to unpack .../libtesseract-dev_4.1.1-2.1build1_amd64.deb ...
Unpacking libtesseract-dev:amd64 (4.1.1-2.1build1) ...
Setting up libleptonica-dev (1.82.0-3build1) ...
Setting up libarchive-dev:amd64 (3.6.0-1ubuntu1.3) ...
Setting up libtesseract-dev:amd64 (4.1.1-2.1build1) ...
Processing triggers for man-db (2.10.2-1) ...


### **Configure the system prompt - what do you need to extract**

In [13]:
SYSTEM_PROMPT = """
You are an expert AI assistant specialized in extracting structured metadata from documents.
The input provided is a JSON object containing the relevant information from which we need to extract content
Analyze the content and the image data to generate a **single, valid JSON object** containing the following fields.
If information for a field is not found, use `null` or an empty string/list as appropriate for the field type.

Extract the following fields:
1.  `title`: The main title of the document. (Type: string)
2.  `journal_conference`: The name of the journal or conference proceedings. (Type: string/null)
3.  `doi`: The Digital Object Identifier. (Type: string/null)
4.  `authors`: A list of author names. (Type: list of strings)
5.  `publication_year`: The year the document was published. (Type: integer/null)
6.  `keywords`: A list of keywords provided in the document. (Type: list of strings)
7.  `abstract`: The abstract section of the document. (Type: string/null)
8.  `summary`: A brief summary generated by you based on the content (especially abstract/conclusions). (Type: string)
9.  `conclusions`: The main conclusions stated in the document. (Type: string/null)
10. `advantages_disadvantages`: Mentioned advantages and disadvantages of the methods/approach. (Type: object with keys "advantages" (list of strings) and "disadvantages" (list of strings))
11. `explainable_ai_mentions`: Specific text snippets discussing Explainable AI (XAI) or interpretability. (Type: list of strings)
12. `future_work_next_steps`: Stated future work or next steps. (Type: list of strings)
13. `region_focus`: Any specific geographical region focus mentioned. (Type: string/null)
14. `dataset_details`: Information about datasets used (e.g., name, size, source). (Type: string/null)
15. `data_types_used`: Types of data analyzed (e.g., text, image, tabular, time-series). (Type: list of strings)
16. `company_types_mentioned`: Types of companies relevant to the study (if any). (Type: list of strings)
17. `time_period_analyzed`: Specific time periods covered by the data or study. (Type: string/null)
18. `models_approach_used`: Key models, algorithms, or approaches employed. (Type: list of strings)
19. `innovative_aspects`: Highlight any particularly innovative algorithms or techniques mentioned. (Type: list of strings)
20. `general_applicability`: Assessment of the general applicability of the findings/methods. (Type: string/null)
21. `performance_metrics`: Key performance results reported for models/methods. (Type: object or list of objects, e.g., {"model": "X", "metric": "Accuracy", "value": "95%"})

Remember: Output **only** the final JSON object. No extra text, explanations, or markdown formatting.
"""

### **App for metadata extraction from documents - PyMuPDF**

In [14]:
# 🚫 DO NOT EDIT – App code and output in Gradio using pymupdf

import gradio as gr
import os
import json
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import io
import pandas as pd
import openai
from datetime import datetime
import tiktoken
import time

DEFAULT_MODEL = "gpt-4o-mini"

# --- PRICING INFORMATION (Update these when prices change) ---
PRICING_DATA = {
    "last_updated": "Apr 2025",
    "prices": {
        "gpt-4o-mini": {"input": 0.00015, "output": 0.0006},
        "gpt-4o": {"input": 0.0025, "output": 0.010},
        "o1-mini": {"input": 0.0011, "output": 0.0044},
        "o3-mini": {"input": 0.0011, "output": 0.0044}
    }
}

# --- TOKEN COUNTING ---
def count_tokens(text, model):
    try:
        encoding = tiktoken.encoding_for_model(model)
        return len(encoding.encode(text))
    except:
        return len(text.split())  # Rough fallback estimate

# --- TEXT + OCR EXTRACTION ---
def extract_text_and_images_from_pdf(pdf_path):
    results = []
    doc = fitz.open(pdf_path)

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text()
        images = page.get_images(full=True)
        image_texts = []

        for img_index, img in enumerate(images, start=1):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            image = Image.open(io.BytesIO(image_bytes))
            ocr_text = pytesseract.image_to_string(image)
            image_texts.append({
                "image_index": img_index,
                "ocr_text": ocr_text.strip()
            })

        results.append({
            "page_num": page_num,
            "text": text.strip(),
            "image_texts": image_texts
        })

    doc.close()
    return results

# --- OPENAI METADATA EXTRACTION ---
def invoke_openai_model(system_prompt, user_content, model, temperature, log):
    try:
        full_prompt = f"{system_prompt.strip()}\n\nIMPORTANT: Respond ONLY in valid JSON. Do NOT include any explanation, commentary, or additional text."

        response = openai.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": full_prompt},
                {"role": "user", "content": user_content},
            ],
            temperature=temperature
        )
        metadata_text = response.choices[0].message.content

        json_start = metadata_text.find('{')
        json_end = metadata_text.rfind('}') + 1
        if json_start == -1 or json_end == -1:
            raise ValueError("❌ No JSON object found in GPT response.")

        metadata_json = json.loads(metadata_text[json_start:json_end])
        return metadata_json

    except Exception as e:
        log.append(f"❌ OpenAI or JSON parsing error: {e}")
        return None

# --- MAIN PIPELINE ---
def run_pipeline(api_key, model, temp, system_prompt, uploaded_files, progress=gr.Progress(track_tqdm=True)):
    start_time = time.time()
    openai.api_key = api_key
    metadata_list = []
    logs = []
    total_input_tokens = 0
    total_output_tokens = 0

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    excel_path = f"metadata_output_{timestamp}.xlsx"
    json_path = f"metadata_output_{timestamp}.json"

    try:
        for file in progress.tqdm(uploaded_files, desc="Processing PDFs"):
            try:
                file_path = file.name
                logs.append(f"📥 Processing file: {file.name}")

                parsed_data = extract_text_and_images_from_pdf(file_path)
                user_input = json.dumps(parsed_data, ensure_ascii=False, indent=2)

                # Count tokens
                doc_tokens = count_tokens(user_input, model)
                prompt_tokens = count_tokens(system_prompt, model)
                logs.append(f"📄 Document tokens: {doc_tokens}")

                metadata = invoke_openai_model(system_prompt, user_input, model, temp, logs)

                if metadata:
                    response_text = json.dumps(metadata, ensure_ascii=False)
                    output_tokens = count_tokens(response_text, model)

                    total_input_tokens += prompt_tokens + doc_tokens
                    total_output_tokens += output_tokens

                    metadata["source_file"] = os.path.basename(file.name)
                    metadata_list.append(metadata)
                    logs.append(f"✅ Done: {file.name} (in:{prompt_tokens+doc_tokens} out:{output_tokens})")
                else:
                    logs.append(f"⚠️ Skipped: {file.name}")

            except Exception as e:
                logs.append(f"❌ Error: {file.name} — {e}")
                continue

        # Calculate time taken
        time_taken = time.time() - start_time
        mins, secs = divmod(time_taken, 60)
        time_str = f"{int(mins)}m {int(secs)}s" if mins > 0 else f"{int(secs)}s"

        # Calculate cost
        total_tokens = total_input_tokens + total_output_tokens
        input_cost = (total_input_tokens / 1000) * PRICING_DATA["prices"][model]["input"]
        output_cost = (total_output_tokens / 1000) * PRICING_DATA["prices"][model]["output"]
        total_cost = input_cost + output_cost

        # Add summary section
        logs.append("\n" + "=" * 50)
        logs.append("SUMMARY:")
        logs.append(f"Time taken: {time_str}")
        logs.append(f"Processed: {len(uploaded_files)} files")
        logs.append(f"Total tokens: {total_tokens} (in:{total_input_tokens} out:{total_output_tokens})")
        logs.append(f"Estimated cost (@{PRICING_DATA['last_updated']}): ${total_cost:.4f}")
        logs.append("=" * 50)

        df = pd.DataFrame(metadata_list)
        df.to_excel(excel_path, index=False)

        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(metadata_list, f, ensure_ascii=False, indent=2)

        return excel_path, json_path, "\n".join(logs)

    except Exception as e:
        logs.append(f"❌ Unexpected pipeline error: {e}")
        return None, None, "\n".join(logs)

# --- GRADIO UI ---
with gr.Blocks(title="📚 AI PDF Metadata Extractor", css="""
    .fixed-log-box textarea {
        overflow-y: auto !important;
        resize: none !important;
    }
""") as demo:
    gr.Markdown("## 📚 AI PDF Metadata Extractor by AI Academy")
    gr.Markdown("Upload your documents, set your prompt, and extract structured metadata with ChatGPT.")

    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            system_prompt = gr.Textbox(label="🧠 System Prompt", lines=18, value=SYSTEM_PROMPT)
            api_key = gr.Textbox(label="🔐 OpenAI API Key", type="password")
            model_choices = list(PRICING_DATA["prices"].keys())
            model = gr.Dropdown(
                choices=model_choices,
                value=DEFAULT_MODEL,
                label="🧠 OpenAI Model"
            )
            temp = gr.Slider(0.0, 1.0, value=0.0, step=0.1, label="🎛️ Temperature")

        with gr.Column(scale=1):
            uploaded = gr.File(label="📄 Upload PDF(s)", file_types=[".pdf"], file_count="multiple", height=150)
            with gr.Row():
                submit_btn = gr.Button("🚀 Extract Metadata", variant="primary")
                clear_btn = gr.Button("🧹 Clear Data")
                reset_btn = gr.Button("🔁 Reset All")

            log_output = gr.Textbox(label="🪵 Processing Logs", lines=8, interactive=False, max_lines=20, show_copy_button=True, elem_classes="fixed-log-box")

            with gr.Group():
                gr.Markdown("### 📥 Download Results")
                with gr.Row():
                    excel_output = gr.File(label="Excel", interactive=False, file_types=[".xlsx"])
                    json_output = gr.File(label="JSON", interactive=False, file_types=[".json"], elem_classes="fixed-download")

    submit_btn.click(
        fn=run_pipeline,
        inputs=[api_key, model, temp, system_prompt, uploaded],
        outputs=[excel_output, json_output, log_output]
    )

    clear_btn.click(
        fn=lambda: (None, None, None, ""),
        inputs=[],
        outputs=[uploaded, excel_output, json_output, log_output]
    )

    reset_btn.click(
        fn=lambda: ("", DEFAULT_MODEL, 0.0, None, None, None, None, ""),
        inputs=[],
        outputs=[api_key, model, temp, uploaded, excel_output, json_output, log_output]
    )

    demo.launch()

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://29dd1e9c4ab02fcfc2.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### **App for metadata extraction from documents - PyMuPDF4LLM**

In [12]:
# @title
# 🚫 DO NOT EDIT – App code and output in Gradio using pymupdf4llm

import gradio as gr
import os
import json
import fitz  # PyMuPDF
import pymupdf4llm # For Markdown conversion
import pytesseract
from PIL import Image
import io
import pandas as pd
import openai
from datetime import datetime
import tiktoken
import time

# --- TESSERACT CONFIGURATION (Optional: Uncomment and set path if needed) ---
# pytesseract.pytesseract.tesseract_cmd = r'/usr/bin/tesseract' # Example for Linux
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe' # Example for Windows

DEFAULT_MODEL = "gpt-4o-mini"

# --- PRICING INFORMATION (Apr 2025) ---
PRICING_DATA = {
    "last_updated": "Apr 2025",
    "prices": {
        "gpt-4o-mini": {"input": 0.00015, "output": 0.0006},
        "gpt-4o": {"input": 0.0025, "output": 0.010},
        "o1-mini": {"input": 0.0011, "output": 0.0044},
        "o3-mini": {"input": 0.0011, "output": 0.0044}
    }
}
model_choices = list(PRICING_DATA["prices"].keys())

# --- TOKEN COUNTING ---
def count_tokens(text, model):
    """Counts tokens using tiktoken or falls back to word count."""
    model_name_for_tiktoken = model
    if model not in ["gpt-4o", "gpt-4o-mini", "gpt-3.5-turbo"]:
         if "gpt-4" in model: model_name_for_tiktoken = "gpt-4"
         elif "gpt-3.5" in model: model_name_for_tiktoken = "gpt-3.5-turbo"
         else: model_name_for_tiktoken = "gpt-4"

    try:
        encoding = tiktoken.encoding_for_model(model_name_for_tiktoken)
        return len(encoding.encode(text))
    except Exception as e:
        return len(text.split())

# --- CONTENT EXTRACTION (using pymupdf4llm + Tesseract) ---
def extract_content_with_pymupdf4llm(pdf_path):
    """Extracts text/tables as Markdown and OCRs images."""
    doc = None
    try:
        md_text = pymupdf4llm.to_markdown(pdf_path, write_images=False)
        image_ocr_results = []
        doc = fitz.open(pdf_path)
        for page_num, page in enumerate(doc, start=1):
            images = page.get_images(full=True)
            page_image_texts = []
            for img_index, img in enumerate(images, start=1):
                xref = img[0]
                try:
                    base_image = doc.extract_image(xref)
                    image_bytes = base_image["image"]
                    image = Image.open(io.BytesIO(image_bytes))
                    if image.mode == 'RGBA':
                        image = image.convert('RGB')
                    ocr_text = pytesseract.image_to_string(image)
                    page_image_texts.append({"image_index": img_index, "ocr_text": ocr_text.strip()})
                except Exception as img_e:
                    print(f"Warning: Could not process image {img_index} on page {page_num} in {os.path.basename(pdf_path)}: {img_e}")
                    page_image_texts.append({"image_index": img_index, "ocr_text": f"Error processing image: {img_e}"})
            if page_image_texts:
                 image_ocr_results.append({"page_num": page_num, "image_texts": page_image_texts})
        doc.close()
        return {"markdown_content": md_text.strip(), "image_ocr_data": image_ocr_results}
    except Exception as e:
        print(f"Error during PDF processing ({os.path.basename(pdf_path)}): {e}")
        if doc:
            try: doc.close()
            except: pass
        return None

# --- OPENAI METADATA EXTRACTION ---
def invoke_openai_model(system_prompt, user_content, model, temperature, log):
    """Invokes OpenAI API and parses JSON response."""
    try:
        full_prompt = f"{system_prompt.strip()}\n\nIMPORTANT: Respond ONLY in valid JSON format. Do NOT include any introductory text, explanations, commentary, markdown formatting (like ```json), or any text outside the main JSON object."
        response = openai.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": full_prompt},
                {"role": "user", "content": user_content},
            ],
            temperature=temperature,
            response_format={"type": "json_object"}
        )
        metadata_text = response.choices[0].message.content
        metadata_json = json.loads(metadata_text)
        return metadata_json
    except json.JSONDecodeError as json_e:
        log.append(f"❌ JSON parsing error: {json_e}. Response snippet:\n{metadata_text[:500]}...")
        try: # Fallback extraction
            json_start = metadata_text.find('{')
            json_end = metadata_text.rfind('}') + 1
            if json_start != -1 and json_end != -1:
                metadata_json = json.loads(metadata_text[json_start:json_end])
                log.append("⚠️ Used fallback JSON extraction.")
                return metadata_json
            else: raise ValueError("No JSON object found even with fallback.")
        except Exception as fallback_e:
             log.append(f"❌ Fallback JSON extraction failed: {fallback_e}")
             return None
    except openai.AuthenticationError:
        log.append("❌ OpenAI Authentication Error: Check API key and credits.")
        raise
    except Exception as e:
        log.append(f"❌ OpenAI API call or other error: {e}")
        return None

# --- MAIN PIPELINE ---
def run_pipeline(api_key, model, temp, system_prompt, uploaded_files, progress=gr.Progress(track_tqdm=True)):
    start_time = time.time()
    if not api_key: return None, None, "❌ Error: OpenAI API Key is missing."
    try: openai.api_key = api_key
    except openai.AuthenticationError: return None, None, "❌ OpenAI Authentication Error: Invalid API Key or insufficient credits."
    except Exception as api_e: return None, None, f"❌ Error initializing OpenAI client: {api_e}"

    metadata_list = []
    logs = ["Pipeline started..."]
    total_input_tokens, total_output_tokens = 0, 0
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    excel_path = f"metadata_output_{timestamp}.xlsx"
    json_path = f"metadata_output_{timestamp}.json"

    if not uploaded_files: return None, None, "⚠️ No files uploaded."
    if model not in PRICING_DATA["prices"]: return None, None, f"❌ Error: Selected model '{model}' not available."

    try:
        for file in progress.tqdm(uploaded_files, desc="Processing PDFs"):
            file_basename = os.path.basename(file.name)
            try:
                logs.append(f"📥 Processing file: {file_basename}")
                extracted_data = extract_content_with_pymupdf4llm(file.name)
                if not extracted_data:
                    logs.append(f"⚠️ Skipped (extraction failed): {file_basename}")
                    continue

                user_input_dict = {"document_markdown": extracted_data["markdown_content"], "images_ocr": extracted_data["image_ocr_data"]}
                user_content_for_api = json.dumps(user_input_dict, ensure_ascii=False, indent=2)

                doc_tokens = count_tokens(user_content_for_api, model)
                prompt_tokens = count_tokens(system_prompt, model)
                current_input_tokens = prompt_tokens + doc_tokens
                logs.append(f"📄 Tokens (Prompt: {prompt_tokens}, Content: {doc_tokens}) = Input: {current_input_tokens}")

                metadata = invoke_openai_model(system_prompt, user_content_for_api, model, temp, logs)
                if metadata and isinstance(metadata, dict):
                    response_text = json.dumps(metadata, ensure_ascii=False)
                    output_tokens = count_tokens(response_text, model)
                    total_input_tokens += current_input_tokens
                    total_output_tokens += output_tokens
                    metadata["source_file"] = file_basename
                    metadata_list.append(metadata)
                    logs.append(f"✅ Done: {file_basename} (In: {current_input_tokens}, Out: {output_tokens})")
                else:
                    logs.append(f"⚠️ Skipped (OpenAI/JSON error or invalid format): {file_basename}")

            except openai.AuthenticationError:
                 logs.append("❌ OpenAI Authentication Error during processing. Stopping.")
                 raise
            except Exception as e:
                logs.append(f"❌ Error processing {file_basename}: {e}")
                import traceback
                logs.append(traceback.format_exc())
                continue

        # --- Summary Calculation ---
        time_taken = time.time() - start_time
        mins, secs = divmod(time_taken, 60)
        time_str = f"{int(mins)}m {int(secs)}s" if mins > 0 else f"{int(secs)}s"
        model_pricing = PRICING_DATA["prices"][model]
        input_cost = (total_input_tokens / 1000) * model_pricing["input"]
        output_cost = (total_output_tokens / 1000) * model_pricing["output"]
        total_cost = input_cost + output_cost

        logs.append("\n" + "=" * 50)
        logs.append("📊 SUMMARY:")
        logs.append(f"⏱️ Time taken: {time_str}")
        logs.append(f"📚 Files processed successfully: {len(metadata_list)} / {len(uploaded_files)}")
        logs.append(f"🤖 Model used: {model}")
        logs.append(f"📈 Total tokens: {total_input_tokens + total_output_tokens} (Input: {total_input_tokens}, Output: {total_output_tokens})")
        logs.append(f"💲 Estimated cost (@{PRICING_DATA['last_updated']}): ${total_cost:.4f}")
        logs.append("=" * 50)

        if not metadata_list:
            logs.append("⚠️ No metadata was successfully extracted.")
            return None, None, "\n".join(logs)

        df = pd.DataFrame(metadata_list)
        df.to_excel(excel_path, index=False)
        logs.append(f"💾 Excel results saved to: {excel_path}")
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(metadata_list, f, ensure_ascii=False, indent=2)
        logs.append(f"💾 JSON results saved to: {json_path}")
        return excel_path, json_path, "\n".join(logs)

    except openai.AuthenticationError:
         logs.append("❌ OpenAI Authentication Error.")
         return None, None, "\n".join(logs)
    except Exception as e:
        logs.append(f"❌ Unexpected pipeline error: {e}")
        import traceback
        logs.append(traceback.format_exc())
        return None, None, "\n".join(logs)


# --- GRADIO UI ---
with gr.Blocks(title="📚 AI PDF Metadata Extractor") as demo:
    gr.Markdown("## 📚 AI PDF Metadata Extractor by AI Academy")
    gr.Markdown("Upload documents, define JSON structure via prompt, select model, and extract metadata.")

    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            system_prompt = gr.Textbox(label="📝 System Prompt (Define JSON structure)", lines=18, value=SYSTEM_PROMPT)
            api_key = gr.Textbox(label="🔐 OpenAI API Key", type="password", placeholder="Enter your OpenAI API key")
            model = gr.Dropdown(choices=model_choices, value=DEFAULT_MODEL, label="🧠 Select OpenAI Model")
            temp = gr.Slider(0.0, 1.0, value=0.0, step=0.1, label="🎛️ Temperature (0=Deterministic, 1=Creative)")

        with gr.Column(scale=1):
            uploaded = gr.File(label="📄 Upload PDF(s)", file_types=[".pdf"], file_count="multiple", height=150)
            with gr.Row():
                submit_btn = gr.Button("🚀 Extract Metadata", variant="primary")
                clear_btn = gr.Button("🧹 Clear Uploads & Logs")
                reset_btn = gr.Button("🔁 Reset All Fields")

            log_output = gr.Textbox(
                label="🪵 Processing Logs",
                lines=8, # Adjust this value if needed for visual balance
                interactive=False,
                max_lines=40, # Limit max lines, but scrolling handles overflow
                show_copy_button=True,
                elem_classes="fixed-log-box"
            )

            with gr.Group():
                gr.Markdown("### 📥 Download Results")
                with gr.Row():
                    excel_output = gr.File(label="Excel (.xlsx)", interactive=False, file_types=[".xlsx"], elem_classes="fixed-download")
                    json_output = gr.File(label="JSON (.json)", interactive=False, file_types=[".json"], elem_classes="fixed-download")

    # --- Event Handlers ---
    submit_btn.click(
        fn=run_pipeline,
        inputs=[api_key, model, temp, system_prompt, uploaded],
        outputs=[excel_output, json_output, log_output]
    )

    clear_btn.click(
        fn=lambda: (None, None, None, ""),
        inputs=[],
        outputs=[uploaded, excel_output, json_output, log_output]
    )

    reset_btn.click(
        fn=lambda: ("", DEFAULT_MODEL, 0.0, None, None, None, ""),
        inputs=[],
        outputs=[api_key, model, temp, uploaded, excel_output, json_output, log_output]
    )

# --- Launch the App ---
if __name__ == "__main__":
    demo.launch(debug=True)

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://feaae2b5fb562c5ca2.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://feaae2b5fb562c5ca2.gradio.live


### **Document Pipeline App including Large Document handling with ChatGPT**

In [10]:
# 🚫 DO NOT EDIT – Gradip App for large document handling

from typing import Dict, List, Union, Tuple
import gradio as gr
import os
import json
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import io
import pandas as pd
import openai
from datetime import datetime
import tiktoken
from concurrent.futures import ThreadPoolExecutor
import time

# Constants
MODEL_PRICING = {
    "gpt-4-turbo": {"input": 0.01, "output": 0.03},
    "gpt-4": {"input": 0.03, "output": 0.06},
    "gpt-3.5-turbo": {"input": 0.0015, "output": 0.002}
}

class DocumentProcessor:
    def __init__(self):
        self.total_cost = 0.0
        self.total_tokens = 0

    def _call_openai(self, messages: List[Dict], model: str, **kwargs) -> Dict:
        """API call with basic retry logic"""
        for attempt in range(3):
            try:
                response = openai.chat.completions.create(
                    model=model,
                    messages=messages,
                    temperature=0,  # Fully deterministic
                    **kwargs
                )
                self._update_cost(response.usage, model)
                return response
            except Exception as e:
                if attempt == 2:
                    raise
                time.sleep(2 ** attempt)  # Exponential backoff

    def _update_cost(self, usage, model):
        """Track token usage and costs"""
        self.total_tokens += usage.total_tokens
        self.total_cost += (
            usage.prompt_tokens * MODEL_PRICING[model]["input"] / 1000 +
            usage.completion_tokens * MODEL_PRICING[model]["output"] / 1000
        )

    def _smart_chunking(self, text: str, model: str) -> List[str]:
        """Chunking that preserves paragraph boundaries"""
        enc = tiktoken.encoding_for_model(model)
        tokens = enc.encode(text)

        chunks = []
        current_chunk = []

        for token in tokens:
            current_chunk.append(token)

            if len(current_chunk) >= 45000:  # Target chunk size with buffer
                chunk_text = enc.decode(current_chunk)
                # Split at last paragraph break
                last_break = max(
                    chunk_text.rfind("\n\n"),
                    chunk_text.rfind(". "),
                    chunk_text.rfind("! "),
                    chunk_text.rfind("? ")
                )

                if last_break > 0:
                    chunks.append(chunk_text[:last_break+1])
                    current_chunk = enc.encode(chunk_text[last_break+1:])
                else:
                    chunks.append(chunk_text)
                    current_chunk = []

        if current_chunk:
            chunks.append(enc.decode(current_chunk))

        return chunks

    def _extract_content(self, pdf_path: str) -> str:
        """Simplified extraction with English-only OCR"""
        doc = fitz.open(pdf_path)
        full_text = []

        for page in doc:
            # Text extraction
            text = page.get_text("text").strip()
            if text:
                full_text.append(text)

            # OCR with English (avoids language detection)
            for img in page.get_images(full=True):
                try:
                    base_img = doc.extract_image(img[0])
                    image = Image.open(io.BytesIO(base_img["image"]))
                    ocr_text = pytesseract.image_to_string(image, lang='eng')
                    full_text.append(ocr_text.strip())
                except Exception:
                    continue

        doc.close()
        return "\n\n".join(full_text)

    def process(self, file_path: str, system_prompt: str, model: str) -> Dict:
        """Core processing pipeline"""
        try:
            content = self._extract_content(file_path)
            if not content.strip():
                return {"error": "No extractable content"}

            chunks = self._smart_chunking(content, model)

            if len(chunks) == 1:
                result = self._get_metadata(chunks[0], system_prompt, model)
            else:
                summaries = []
                for chunk in chunks:
                    summary = self._summarize_chunk(chunk, system_prompt, model)
                    summaries.append(summary)
                result = self._get_metadata("\n\n".join(summaries), system_prompt, model)

            result["source_file"] = os.path.basename(file_path)
            return {
                "title": result.get("title", "Untitled"),
                "content": result.get("content", ""),
                **{k: v for k, v in result.items() if k not in ["title", "content"]}
            }

        except Exception as e:
            return {"error": str(e), "source_file": os.path.basename(file_path)}

    def _summarize_chunk(self, chunk: str, prompt: str, model: str) -> str:
        response = self._call_openai(
            messages=[
                {"role": "system", "content": "Create concise summary preserving key facts"},
                {"role": "user", "content": f"{prompt}\n\n{chunk}"}
            ],
            model=model
        )
        return response.choices[0].message.content

    def _get_metadata(self, text: str, prompt: str, model: str) -> Dict:
        response = self._call_openai(
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": text}
            ],
            model=model,
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)

def run_pipeline(api_key: str, model: str, system_prompt: str, files: List[str]) -> Tuple[str, str, str]:
    """Batch processing with cost tracking"""
    openai.api_key = api_key
    processor = DocumentProcessor()
    results = []

    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = [executor.submit(processor.process, f.name, system_prompt, model) for f in files]
        results = [f.result() for f in futures]

    # Generate outputs
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    excel_path = f"results_{timestamp}.xlsx"
    json_path = f"results_{timestamp}.json"

    pd.DataFrame(results).to_excel(excel_path, index=False)
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)

    log = (
        f"Processed {len(results)} files\n"
        f"Total tokens: {processor.total_tokens}\n"
        f"Estimated cost: ${processor.total_cost:.4f}"
    )

    return excel_path, json_path, log

with gr.Blocks(title="PRO Document Processor") as app:
    gr.Markdown("## 🔍 Ultimate Document Processor")

    with gr.Row():
        with gr.Column():
            api_key = gr.Textbox(label="OpenAI API Key", type="password")
            model = gr.Dropdown(
                list(MODEL_PRICING.keys()),
                value="gpt-4-turbo",
                label="Model"
            )
            prompt = gr.Textbox(
                label="System Prompt",
                value="Extract ALL metadata in JSON format...",
                lines=8
            )
            files = gr.File(file_count="multiple", file_types=[".pdf"])

        with gr.Column():
            submit = gr.Button("Process", variant="primary")
            with gr.Group():
                gr.Markdown("### Output")
                excel = gr.File(label="Excel")
                json_out = gr.File(label="JSON")
                log = gr.Textbox(label="Processing Log", interactive=False)

            gr.Markdown("### Cost Estimate")
            cost = gr.Textbox(label="Estimated Cost", interactive=False)

    submit.click(
        fn=run_pipeline,
        inputs=[api_key, model, prompt, files],
        outputs=[excel, json_out, log]
    )

app.launch()

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://227531f18494fade1e.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### **Batch Processing of documents from Drive with ChatGPT**

#### Configure batch processing parameters

In [ ]:
from google.colab import userdata

# 🚀 USER CONFIGURATION (EDIT THIS CELL ONLY)

# 📂 Root folder in Google Drive (must contain Input Docs / Output Docs / Results)
BASE_PATH = '/content/drive/My Drive/_Profi/_AI L&D/Docs'

# 🔁 Set to True to force reprocess all files, even if metadata already exists
RUN_ALL = False

# 🔐 OpenAI Settings
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
OPENAI_MODEL = 'gpt-4o'
OPENAI_TEMPERATURE = 0.0
OPENAI_MAX_TOKENS = 4096  # Optional: for controlling response size

#### Pre-process documents to extract relevant content

In [ ]:
# @title
# 🚫 DO NOT EDIT – PDF to JSON parsing (text + image OCR)

import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import io
import os, json
import openai
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Derived paths
INPUT_DOCS = os.path.join(BASE_PATH, 'Input Docs')
OUTPUT_DOCS = os.path.join(BASE_PATH, 'Output Docs')
RESULTS_FOLDER = os.path.join(BASE_PATH, 'Results')
os.makedirs(OUTPUT_DOCS, exist_ok=True)
os.makedirs(RESULTS_FOLDER, exist_ok=True)

METADATA_JSON = os.path.join(RESULTS_FOLDER, 'Consolidated_Metadata.json')
METADATA_XLSX = os.path.join(RESULTS_FOLDER, 'Consolidated_Metadata.xlsx')

# Configure OpenAI
openai.api_key = OPENAI_API_KEY

def extract_text_and_images_from_pdf(pdf_path):
    """Extract native text and OCR text from images inside a PDF."""
    results = []
    doc = fitz.open(pdf_path)

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text()

        images = page.get_images(full=True)
        image_texts = []

        for img_index, img in enumerate(images, start=1):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            image = Image.open(io.BytesIO(image_bytes))

            # OCR with pytesseract
            ocr_text = pytesseract.image_to_string(image)
            image_texts.append({
                "image_index": img_index,
                "ocr_text": ocr_text.strip()
            })

        results.append({
            "page_num": page_num,
            "text": text.strip(),
            "image_texts": image_texts
        })

    doc.close()
    return results

def save_results_to_json(results, filename, destination_folder):
    """Save extracted results to a JSON file in the Output Docs folder."""
    os.makedirs(destination_folder, exist_ok=True)
    json_filename = os.path.splitext(os.path.basename(filename))[0] + ".json"
    json_path = os.path.join(destination_folder, json_filename)

    with open(json_path, "w", encoding="utf-8") as json_file:
        json.dump(results, json_file, ensure_ascii=False, indent=4)

    print(f"✅ Saved parsed JSON: {json_path}")
    return json_filename

def process_pdfs_from_directory(source_folder, output_folder, results_folder):
    """Process PDFs from Input Docs and save JSONs + summary Excel."""
    summary = []
    os.makedirs(results_folder, exist_ok=True)

    for filename in os.listdir(source_folder):
        if not filename.lower().endswith(".pdf"):
            continue

        json_filename = os.path.splitext(filename)[0] + ".json"
        json_path = os.path.join(output_folder, json_filename)

        if os.path.exists(json_path):
            print(f"⏭️ Skipping {filename}: already parsed.")
            continue

        pdf_path = os.path.join(source_folder, filename)
        results = extract_text_and_images_from_pdf(pdf_path)
        save_results_to_json(results, filename, output_folder)

        summary.append({
            "pdf_file": filename,
            "json_file": json_filename,
            "pages": len(results),
        })

    # Save summary to Excel
    if summary:
        df_summary = pd.DataFrame(summary)
        excel_path = os.path.join(results_folder, "processing_summary.xlsx")
        df_summary.to_excel(excel_path, index=False)
        print(f"\n📊 Summary saved to Excel: {excel_path}")
    else:
        print("\n📂 No new PDFs processed.")

# ✅ RUN THIS to parse any new PDFs in Input Docs
process_pdfs_from_directory(
    source_folder=INPUT_DOCS,
    output_folder=OUTPUT_DOCS,
    results_folder=RESULTS_FOLDER
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⏭️ Skipping The significance of financial and non-financial information in  insolvency risk detection.pdf: already parsed.
⏭️ Skipping PREDICTION OF CORPORATE BANKRUPTCY IN ROMANIA THROUGH THE USE OF LOGISTIC REGRESSION.pdf: already parsed.
⏭️ Skipping BALANCED BAGGING WITH EXPECTATION MAXIMIZATION.pdf: already parsed.
⏭️ Skipping Diagnostic model of the risk of bankruptcy.pdf: already parsed.

📂 No new PDFs processed.


#### Extract metadata with ChatGPT based on prompt instructions

In [ ]:
# @title
# 🚫 DO NOT EDIT – SETUP: Mount Drive, prepare folders and OpenAI call + metadata logic

def invoke_openai_model(system_prompt, user_content):
    """Call OpenAI GPT model with system prompt and content."""
    try:
        response = openai.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content},
            ],
            temperature=OPENAI_TEMPERATURE
        )
        metadata_text = response.choices[0].message.content
        metadata_json = json.loads(metadata_text[metadata_text.find('{'):metadata_text.rfind('}') + 1])
        return metadata_json
    except Exception as e:
        print(f"❌ OpenAI Error: {e}")
        return None

def load_existing_metadata(path):
    if os.path.exists(path):
        try:
            with open(path, "r", encoding="utf-8") as file:
                return json.load(file)
        except:
            print("⚠️ Metadata file exists but could not be read. Starting fresh.")
    return []

# 🚫 DO NOT EDIT – Metadata extraction pipeline

def process_json_files(source_folder, metadata_path, run_all=False):
    """Process OCR+text JSONs and extract metadata with OpenAI."""
    if run_all and os.path.exists(metadata_path):
        os.remove(metadata_path)
        print("🗑️ Deleted existing metadata — reprocessing all files.")

    consolidated = load_existing_metadata(metadata_path)
    already_processed = {entry.get("source_file") for entry in consolidated if "source_file" in entry}

    processed_count = 0
    skipped_count = 0
    failed_files = []

    for filename in os.listdir(source_folder):
        filepath = os.path.join(source_folder, filename)

        if not filename.endswith(".json") or not os.path.isfile(filepath):
            continue
        if not run_all and filename in already_processed:
            print(f"⏭️ Already processed: {filename}")
            skipped_count += 1
            continue

        try:
            with open(filepath, "r", encoding="utf-8") as file:
                doc_content = json.load(file)
            user_input = json.dumps(doc_content, ensure_ascii=False, indent=2)

            print(f"\n📄 Extracting metadata for: {filename}")
            metadata = invoke_openai_model(SYSTEM_PROMPT, user_input)

            if metadata:
                metadata["source_file"] = filename
                consolidated.append(metadata)

                with open(metadata_path, "w", encoding="utf-8") as file:
                    json.dump(consolidated, file, ensure_ascii=False, indent=4)

                print(f"✅ Saved: {filename}")
                processed_count += 1
            else:
                failed_files.append(filename)

        except Exception as e:
            print(f"⚠️ Failed to process {filename}: {e}")
            failed_files.append(filename)

    print(f"\n✅ Done: {processed_count} processed | ⏭️ {skipped_count} skipped | ❌ {len(failed_files)} failed")
    return pd.DataFrame(consolidated)

# ✅ RUN THIS to execute the metadata extraction
df_metadata = process_json_files(
    source_folder=OUTPUT_DOCS,
    metadata_path=METADATA_JSON,
    run_all=RUN_ALL
)

if not df_metadata.empty:
    df_metadata.to_excel(METADATA_XLSX, index=False)
    print(f"\n📊 Metadata saved to Excel: {METADATA_XLSX}")
    display(df_metadata.head())
else:
    print("⚠️ No metadata extracted.")


⏭️ Already processed: The significance of financial and non-financial information in  insolvency risk detection.json
⏭️ Already processed: PREDICTION OF CORPORATE BANKRUPTCY IN ROMANIA THROUGH THE USE OF LOGISTIC REGRESSION.json
⏭️ Already processed: BALANCED BAGGING WITH EXPECTATION MAXIMIZATION.json
⏭️ Already processed: Diagnostic model of the risk of bankruptcy.json

✅ Done: 0 processed | ⏭️ 4 skipped | ❌ 0 failed

📊 Metadata saved to Excel: /content/drive/My Drive/_Profi/_AI L&D/Docs/Results/Consolidated_Metadata.xlsx


,Title,Journal,DOI,Authors,Publication Year,Keywords,Abstract,Summary,Conclusions,Advantages/Disadvantages,...,Region,Dataset Size,Types of Data Used,Types of Companies,Period Analyzed,Models/Approach Used,Innovative Algorithms,General Applicability,Performance for Each Model,source_file
0,The significance of financial and non-financia...,Procedia Economics and Finance,10.1016/S2212-5671(15)00834-5,"[Mironiuc Marilena, Taran Alina]",2015,"[insolvency, financial information, multiple d...",Insolvency places under uncertainty the premis...,The study investigates the role of financial a...,The study shows the prevalence of financial in...,{'Advantages': 'The study provides a comprehen...,...,[Romania],20 companies,"[financial, non-financial]",[listed companies],2009-2013,"[multiple discriminant analysis, logistic regr...",No,The study's findings are applicable to underst...,"[{'model': 'Multiple Discriminant Analysis', '...",The significance of financial and non-financia...
1,Prediction of Corporate Bankruptcy in Romania ...,Not specified,Not specified,"[Brîndescu Daniel, Goleț Ionuț]",2023,"[bankruptcy, financial statement analysis, eco...",The purpose of this paper is to test whether d...,The study evaluates the use of logistic regres...,"The fixed assets ratio, fixed assets turnover ...",{'Advantages': 'The model provides a practical...,...,[Romania],"4,327 companies",[financial],"[SMEs, large enterprises]",2008-2012,[Logistic Regression],No,The model is expected to maintain its accuracy...,"[{'model': 'Logistic Regression', 'in-sample a...",PREDICTION OF CORPORATE BANKRUPTCY IN ROMANIA ...
2,Balanced Bagging with Expectation Maximization...,Revista Economica,,[Claudiu Clement],2022,"[bankruptcy, machine learning, classification]",Bankruptcy prediction models are widely used b...,This paper investigates the effect of two impu...,The experimental results show that the Expecta...,{'Advantages': 'Balanced Bagging with Expectat...,...,[Romania],"More than 20,000 companies",[financial],[Romanian companies],2016-2019,"[Balanced Bagging, Logistic Regression, Decisi...",Yes,The study shows that financial statements data...,[{'model': 'Balanced Bagging with Expectation ...,BALANCED BAGGING WITH EXPECTATION MAXIMIZATION...
3,Diagnostic model of the risk of bankruptcy,Procedia Economics and Finance,10.1016/S2212-5671(14)00632-7,[Bircea Ioana],2014,"[Diagnosis, Insolvency, Risks]","In Romania, on the background of the economic ...",The study develops a financial diagnosis model...,Financial diagnosis submitted is checked only ...,{'Advantages': 'The model provides a quick and...,...,[Romania],34 companies,[financial],[small companies],Not specified,"[Financial diagnosis model, Score-based analysis]",No,The model is applicable to small companies in ...,[],Diagnostic model of the risk of bankruptcy.json
